In [13]:

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.firefox.options import Options
from selenium.webdriver import Firefox
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException
from selenium.common.exceptions import WebDriverException
from selenium import webdriver
from selenium.webdriver.edge.service import Service
#from selenium.webdriver.edge.options import Options




from time import sleep
import warnings
import requests
import pandas as pd
from bs4 import BeautifulSoup
import time
import math
import re

import sqlite3


from datetime import datetime

import numpy as np
import pyodbc as odbc
from sqlalchemy import create_engine
import sqlalchemy as sa

warnings.filterwarnings('ignore')


In [14]:
print(odbc.drivers())

['SQL Server', 'Microsoft Access Driver (*.mdb, *.accdb)', 'Microsoft Excel Driver (*.xls, *.xlsx, *.xlsm, *.xlsb)', 'Microsoft Access Text Driver (*.txt, *.csv)']


In [20]:
# Get the current date
current_date = datetime.now().strftime("%d_%m_%Y")  
cd = datetime.now().strftime("%m_%d_%Y") 
# Capture the start time for postings scraping
start_time_scraping_CG_posts = datetime.now()

# Use EdgeOptions instead of the general Options
# options = Options()
options = webdriver.EdgeOptions()
driver = webdriver.Edge(executable_path='C:/Windows/System32/msedgedriver.exe',options=options)

# Correctly set the binary location for Edge
# options.binary_location = r'C:\Windows\System32\msedge.exe'  # Update with your actual Edge path

options.add_argument("--window-size=1920x1080")
options.add_argument("--headless")
options.add_argument("--verbose")

# Explicitly set the Edge binary location and path to msedgedriver
# service = Service(executable_path=r'C:\Windows\System32\msedgedriver.exe')  # Update with your actual msedgedriver path
# browser = webdriver.Edge(service=service, options=options)
driver.set_page_load_timeout(30)

site_urls, site_locations = [], []


URL = f'https://geo.craigslist.org/iso/us'
driver.get(URL)

all_content = driver.find_elements(By.CSS_SELECTOR, 'ul.height6.geo-site-list')
for i in all_content:
    links = i.find_elements(By.TAG_NAME, 'a')
    for link in links:
        site_urls.append(link.get_attribute('href') if link else None)
    
        site_locations.append(link.text)
    

data = []

try:
    for url in site_urls:
        URL_1 = f'{url}search/cta?bundleDuplicates=1&postedToday=1'
        driver.get(URL_1)
        time.sleep(5)  # Wait for the page to load

        try:
            listings_city_element = driver.find_element(By.CLASS_NAME, 'cl-left-group')
            site_name = listings_city_element.text.splitlines()[1]

            total_listings_element = driver.find_element(By.CLASS_NAME, 'cl-page-number')
            total_listings = int(total_listings_element.text.split('of')[-1].replace(',', ''))
            print(f'Total {total_listings} Listings are observed  in {site_name}')
            total_pages = (total_listings // 120) + 1
            print(f'Total {total_pages} Pages observed in {site_name}')

            for page_number in range(1, total_pages + 1):
                URL = f'{URL_1}#search=1~gallery~{page_number}~0'
                driver.get(URL)
                print(URL)
                time.sleep(10)
                all_content = driver.find_elements(By.CSS_SELECTOR, 'li.cl-search-result.cl-search-view-mode-gallery')
                
                for i in all_content:
                    title_element = i.find_element(By.CSS_SELECTOR, 'span.label') if i.find_elements(By.CSS_SELECTOR, 'span.label') else None
                    price_element = i.find_element(By.CLASS_NAME, 'priceinfo') if i.find_elements(By.CLASS_NAME, 'priceinfo') else None
                    tag_element = i.find_element(By.CLASS_NAME, 'meta') if i.find_elements(By.CLASS_NAME, 'meta') else None
                    
                    try:
                        link_element = i.find_element(By.TAG_NAME, 'a')
                        url = link_element.get_attribute('href') if link_element else None
                    except NoSuchElementException:
                        print(f"No link found in the element, skipping: {title_element}")
                    data.append({
                        'City': site_name,
                        'URL': url,
                        'Title': title_element.text if title_element else None,
                        'Price': price_element.text if price_element else None,
                        'Tags': tag_element.text if tag_element else None
                    })
                    
                    
        except Exception as e:
            print(f"An error occurred: {e}")

finally:
    driver.quit()

# Compile the scraped data into a pandas DataFrame
postings_df = pd.DataFrame(data)

# Capture the end time and calculate duration
end_time_scraping_CG_posts = datetime.now()
duration = (end_time_scraping_CG_posts - start_time_scraping_CG_posts).total_seconds()
hours, remainder = divmod(duration, 3600)
minutes, seconds = divmod(remainder, 60)

# Output the timing information
print(f"Posts Scraping completed in: {int(hours)} hours, {int(minutes)} minutes, and {int(seconds)} seconds.")


Total 60 Listings are observed  in abilene
Total 1 Pages observed in abilene
https://abilene.craigslist.org/search/cta?bundleDuplicates=1&postedToday=1#search=1~gallery~1~0
Total 47 Listings are observed  in akron-canton
Total 1 Pages observed in akron-canton
https://akroncanton.craigslist.org/search/cta?bundleDuplicates=1&postedToday=1#search=1~gallery~1~0
Total 60 Listings are observed  in albany, GA
Total 1 Pages observed in albany, GA
https://albanyga.craigslist.org/search/cta?bundleDuplicates=1&postedToday=1#search=1~gallery~1~0
Total 37 Listings are observed  in albany, NY
Total 1 Pages observed in albany, NY
https://albany.craigslist.org/search/cta?bundleDuplicates=1&postedToday=1#search=1~gallery~1~0
Total 113 Listings are observed  in albuquerque
Total 1 Pages observed in albuquerque
https://albuquerque.craigslist.org/search/cta?bundleDuplicates=1&postedToday=1#search=1~gallery~1~0
Total 60 Listings are observed  in altoona
Total 1 Pages observed in altoona
https://altoona.cra

WebDriverException: Message: disconnected: not connected to DevTools
  (failed to check if window was closed: disconnected: not connected to DevTools)
  (Session info: MicrosoftEdge=126.0.2592.102)
Stacktrace:
	GetHandleVerifier [0x010236A3+37027]
	Microsoft::Applications::Events::time_ticks_t::time_ticks_t [0x00EEFD26+400614]
	Microsoft::Applications::Events::ILogConfiguration::operator* [0x00CF6DA6+3558]
	Microsoft::Applications::Events::ILogConfiguration::GetModules [0x00CE90A7+11303]
	Microsoft::Applications::Events::ILogConfiguration::GetModules [0x00CE8F09+10889]
	Microsoft::Applications::Events::ILogConfiguration::operator* [0x00CF8B10+11088]
	Microsoft::Applications::Events::GUID_t::GUID_t [0x00D548BE+280222]
	Microsoft::Applications::Events::GUID_t::GUID_t [0x00D3FEB6+195734]
	Microsoft::Applications::Events::GUID_t::GUID_t [0x00D235B3+78739]
	Microsoft::Applications::Events::GUID_t::GUID_t [0x00D225A5+74629]
	Microsoft::Applications::Events::GUID_t::GUID_t [0x00D22FBD+77213]


In [ ]:
postings_df.City.nunique()

412

In [ ]:
len(postings_df)

28710

In [ ]:
# Define a function to normalize city names and URLs for comparison
def normalize_text(text):
    # Remove non-alphabetic characters and convert to lowercase
    return re.sub(r'[^a-zA-Z]', '', text).lower()

# Mapping of special cases where city names don't match URL parts directly
special_cases = {
    'birminghamal': 'bham', 
    'cumberlandval': 'chambersburg', 
    'bowlinggreen': 'bgky', 
    'bloomingtonil': 'bn', 
    'clovisportales': 'clovis',
    'deepeasttx':'nacogdoches',
    'easternct':'newlondon', 
    'easternwv':'martinsburg',
        'hawaii':'honolulu', 
            'tricitieswa':'kpr',
     'kenosharacine':'racine',
         'killeentemple':'killeen',           
    'fayettevillear':'fayar',
    'lakeofozarks':'loz',
    'limafindlay':'limaohio',
    'manhattan':'ksu',
    'newhampshire':'nh',
    'northdakota':'nd',
    'northwestks':'nwks',
    'northwestct':'nwct',
    'northwestga':'nwga',
    'northwestok':'enid',
    'potsdammassena':'potsdam',
    'pullmanmoscow':'pullman',
    'rhodeisland':'providence',
    'sanluisobispo':'slo',
    'southdakota':'sd',
    'southeastak':'juneau',
    'southflorida':'miami',
    'southwestmn':'marshall',
    'southwestms':'natchez',
    'southwesttx':'bigbend',
    'yoopers':'up',
    'visaliatulare':'visalia',
    'westernil':'quincy',
    'westvirginia':'wv',
    'heartlandfl':'cfl',
    'highrockies':'rockies',
    'jacksonmi':'jxn',
    'jacksonvillenc':'onslow',
    'northeastsd':'nesd',
    'northernmi':'nmi',
    'northernwv':'wheeling',
    'centralsd':'csd',
    'rochestermn':'rmn',
    'southeastia':'ottumwa',
    'southeastks':'seks',
    'southeastmo':'semo',
    'southernil':'carbondale',
    'southernmd':'smd',
    'southernwv':'swv',
    'southwestks':'swks',
    'southwestmi':'swmi',
    'southwestva':'swva',
    'statecollege':'pennstate',
    'westernky':'westky'
    
    
}


# Define a function to normalize city names and URLs for comparison
def normalize_text(text):
    # Remove non-alphabetic characters and convert to lowercase
    return re.sub(r'[^a-zA-Z]', '', text).lower()

# Define a function that checks if parts of the city in the 'City' column are in the 'URL'
def city_in_url(row):
    # Normalize the city name from the 'City' column
    city_parts = [normalize_text(part) for part in row['City'].split()]
    
    # Normalize the city name from the 'City' column
    city_normalized = normalize_text(row['City']) 
    
    # Check if the city has a special mapping and use it if available
    if city_normalized in special_cases:
        city_normalized = special_cases[city_normalized]

    # Extract and normalize the city part from the URL
    url_part = normalize_text(row['URL'].split("//")[1].split(".")[0])

    # Check if any key part of the city name is present in the URL part
    return (any(city_part in url_part for city_part in city_parts) | (city_normalized in url_part))


# Apply the function to each row
# This creates a boolean Series where True indicates a match between City and URL
matches = postings_df.apply(city_in_url, axis=1)

# Create a new DataFrame with rows where the city matches the URL
postings_df_matched_rows = postings_df[matches]

postings_df_matched_rows = postings_df_matched_rows.drop_duplicates().reset_index()

# Use the formatted date in the filename
postings_filename = f'W:/Personal Projects/UCPP/Notebooks/Data/postings/posting_df_{current_date}.csv'

# Save the DataFrame to CSV with the dynamic filename
postings_df_matched_rows.to_csv(postings_filename)

In [ ]:
"""

# Get the current date
current_date = datetime.now().strftime("%d_%m_%Y")  # Format the date as Day_Month_Year
cd= datetime.now().strftime("%m_%d_%Y") 
# Capture the start time for postings scraping
start_time_scraping_CG_posts = datetime.now()

options = Options()
options.add_argument("--window-size=1920x1080")
options.add_argument("--headless")  # Headless mode
options.add_argument("--verbose")

browser = Firefox(options=options)
browser.set_page_load_timeout(30)

site_urls, site_locations = [], []


URL = f'https://geo.craigslist.org/iso/us'
browser.get(URL)

all_content = browser.find_elements(By.CSS_SELECTOR, 'ul.height6.geo-site-list')
for i in all_content:
    links = i.find_elements(By.TAG_NAME, 'a')
    for link in links:
        site_urls.append(link.get_attribute('href') if link else None)
    
        site_locations.append(link.text)
    

data = []

try:
    for url in site_urls:
        URL_1 = f'{url}search/cta?bundleDuplicates=1&postedToday=1'
        browser.get(URL_1)
        time.sleep(5)  # Wait for the page to load

        try:
            listings_city_element = browser.find_element(By.CLASS_NAME, 'cl-left-group')
            site_name = listings_city_element.text.splitlines()[1]

            total_listings_element = browser.find_element(By.CLASS_NAME, 'cl-page-number')
            total_listings = int(total_listings_element.text.split('of')[-1].replace(',', ''))
            print(f'Total {total_listings} Listings are observed  in {site_name}')
            total_pages = (total_listings // 120) + 1
            print(f'Total {total_pages} Pages observed in {site_name}')

            for page_number in range(1, total_pages + 1):
                URL = f'{URL_1}#search=1~gallery~{page_number}~0'
                browser.get(URL)
                print(URL)
                time.sleep(10)
                all_content = browser.find_elements(By.CSS_SELECTOR, 'li.cl-search-result.cl-search-view-mode-gallery')
                
                for i in all_content:
                    title_element = i.find_element(By.CSS_SELECTOR, 'span.label') if i.find_elements(By.CSS_SELECTOR, 'span.label') else None
                    price_element = i.find_element(By.CLASS_NAME, 'priceinfo') if i.find_elements(By.CLASS_NAME, 'priceinfo') else None
                    tag_element = i.find_element(By.CLASS_NAME, 'meta') if i.find_elements(By.CLASS_NAME, 'meta') else None
                    
                    try:
                        link_element = i.find_element(By.TAG_NAME, 'a')
                        url = link_element.get_attribute('href') if link_element else None
                    except NoSuchElementException:
                        print(f"No link found in the element, skipping: {title_element}")
                    data.append({
                        'City': site_name,
                        'URL': url,
                        'Title': title_element.text if title_element else None,
                        'Price': price_element.text if price_element else None,
                        'Tags': tag_element.text if tag_element else None
                    })
                    
                    
        except Exception as e:
            print(f"An error occurred: {e}")

finally:
    browser.quit()

# Compile the scraped data into a pandas DataFrame
postings_df = pd.DataFrame(data)

# Capture the end time and calculate duration
end_time_scraping_CG_posts = datetime.now()
duration = (end_time_scraping_CG_posts - start_time_scraping_CG_posts).total_seconds()
hours, remainder = divmod(duration, 3600)
minutes, seconds = divmod(remainder, 60)

# Output the timing information
print(f"Posts Scraping completed in: {int(hours)} hours, {int(minutes)} minutes, and {int(seconds)} seconds.")

"""
